In [1]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

project_root = Path.cwd().parent
raw_data_dir = project_root / "data" / "raw"

print(raw_data_dir)
print(raw_data_dir.exists())

c:\seizure-detection-ML-project\data\raw
True


In [2]:
chb01_dir = raw_data_dir / "chb01"
edf_files = sorted(chb01_dir.glob("*.edf"))
print("Number of EDF files: ", len(edf_files))

for file in edf_files[:5]:
    print(file.name)


for file in sorted(chb01_dir.iterdir()):
    print(file.name)

Number of EDF files:  3
chb01_01.edf
chb01_02.edf
chb01_03.edf
chb01-summary.txt
chb01_01.edf
chb01_02.edf
chb01_03.edf


In [3]:
summary_file = chb01_dir / "chb01-summary.txt"
summary_text = summary_file.read_text()
print(summary_text[:2000])

Data Sampling Rate: 256 Hz
*************************

Channels in EDF Files:
**********************
Channel 1: FP1-F7
Channel 2: F7-T7
Channel 3: T7-P7
Channel 4: P7-O1
Channel 5: FP1-F3
Channel 6: F3-C3
Channel 7: C3-P3
Channel 8: P3-O1
Channel 9: FP2-F4
Channel 10: F4-C4
Channel 11: C4-P4
Channel 12: P4-O2
Channel 13: FP2-F8
Channel 14: F8-T8
Channel 15: T8-P8
Channel 16: P8-O2
Channel 17: FZ-CZ
Channel 18: CZ-PZ
Channel 19: P7-T7
Channel 20: T7-FT9
Channel 21: FT9-FT10
Channel 22: FT10-T8
Channel 23: T8-P8

File Name: chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0

File Name: chb01_02.edf
File Start Time: 12:42:57
File End Time: 13:42:57
Number of Seizures in File: 0

File Name: chb01_03.edf
File Start Time: 13:43:04
File End Time: 14:43:04
Number of Seizures in File: 1
Seizure Start Time: 2996 seconds
Seizure End Time: 3036 seconds

File Name: chb01_04.edf
File Start Time: 14:43:12
File End Time: 15:43:12
Number of Seizures in File: 1
S

In [4]:
start = summary_text.find("File Name: chb01_01.edf")
end = summary_text.find("File Name: chb01_04.edf")
sample_section = summary_text[start:end]
print(sample_section)

File Name: chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0

File Name: chb01_02.edf
File Start Time: 12:42:57
File End Time: 13:42:57
Number of Seizures in File: 0

File Name: chb01_03.edf
File Start Time: 13:43:04
File End Time: 14:43:04
Number of Seizures in File: 1
Seizure Start Time: 2996 seconds
Seizure End Time: 3036 seconds




In [ ]:
recording_blocks = summary_text.split("File Name: ")

print("Number of recording blocks: ", len(recording_blocks))
print(recording_blocks[1])


Number of recording blocks:  43
Data Sampling Rate: 256 Hz
*************************

Channels in EDF Files:
**********************
Channel 1: FP1-F7
Channel 2: F7-T7
Channel 3: T7-P7
Channel 4: P7-O1
Channel 5: FP1-F3
Channel 6: F3-C3
Channel 7: C3-P3
Channel 8: P3-O1
Channel 9: FP2-F4
Channel 10: F4-C4
Channel 11: C4-P4
Channel 12: P4-O2
Channel 13: FP2-F8
Channel 14: F8-T8
Channel 15: T8-P8
Channel 16: P8-O2
Channel 17: FZ-CZ
Channel 18: CZ-PZ
Channel 19: P7-T7
Channel 20: T7-FT9
Channel 21: FT9-FT10
Channel 22: FT10-T8
Channel 23: T8-P8




In [13]:
recording_blocks = summary_text.split("File Name: ")[1:]
print("Number of recording blocks:", len(recording_blocks))
print(recording_blocks[0])

Number of recording blocks: 42
chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0




In [14]:
block = recording_blocks[0]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())


print("Filename:", filename)
print("Seizure count:", seizureCount)

Filename: chb01_01.edf
Seizure count: 0


In [15]:
block = recording_blocks[2]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())


print("Filename:", filename)
print("Seizure count:", seizureCount)

Filename: chb01_03.edf
Seizure count: 1


In [19]:
seizure_starts = []
seizure_ends = []
block = recording_blocks[2]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())

    if line.startswith("Seizure Start Time: "):
        startTime = line.split(":", 1)[1].strip()
        startTime = startTime.replace(" seconds", "")
        startTime = int(startTime)
        seizure_starts.append(startTime)

    if line.startswith("Seizure End Time: "):
        endTime = line.split(":", 1)[1].strip()
        endTime = endTime.replace(" seconds", "")
        endTime = int(endTime)
        seizure_ends.append(endTime)

print("Filename:", filename)
print("Seizure count:", seizureCount)
print("Starts:", seizure_starts)
print("Ends:", seizure_ends)

seizure_intervals = list(zip(seizure_starts, seizure_ends))
print(seizure_intervals[0])

Filename: chb01_03.edf
Seizure count: 1
Starts: [2996]
Ends: [3036]
(2996, 3036)
